In [1]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation (Latest GPU-enabled packages)
# -------------------------------------------------------------------------
!pip install -q -U "bitsandbytes>=0.46.1" transformers peft accelerate evaluate rouge-score nltk bert-score sentence-transformers
!pip uninstall -y torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.1 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Model 7 Stand-Alone (No-RAG) Ultimate Benchmark Evaluation
#    MIMIC-CXR Internal Test Set (N = 6,388 Cases)
#    Pure Stand-Alone Architecture: Prompt has "Reference: None."
#    Beam Search (num_beams=4) | 2D Spatial Layout (16x8)
#    Perceiver Resampler (32 Queries) + Soft Classifier Conditioning
#    Complete 360-Degree Evaluation: NLP + CheXpert F1 + Clinical BERTScore + SBERT
# -------------------------------------------------------------------------
import os, sys, gc, glob, time, json, math, random, re, torch
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# CUDA memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# -------------------------------------------------------------------------
# Execution Configuration: Set QUICK_TEST = False for full 6,388 benchmark
# -------------------------------------------------------------------------
QUICK_TEST = False  # Set to True (or an integer like 20) for a 30-second dry run

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.util import ngrams
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, set_peft_model_state_dict

# -------------------------------------------------------------------------
# Fix: Bypass Kaggle torchao version mismatch bug in PEFT
# -------------------------------------------------------------------------
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
try:
    import peft.tuners.lora.torchao as _torchao_tuner
    _torchao_tuner.is_torchao_available = lambda: False
except Exception:
    pass
try:
    import peft.tuners.lora.model as _lora_model
    if hasattr(_lora_model, "dispatchers"):
        _lora_model.dispatchers = [d for d in _lora_model.dispatchers if "torchao" not in getattr(d, "__name__", "")]
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluation Compute Device: {device}", flush=True)

# -------------------------------------------------------------------------
# Locate Test Data, Checkpoints & Thresholds (Priority to Best Checkpoint)
# -------------------------------------------------------------------------
print("Scanning mounted input directories in /kaggle/input and /kaggle/working...", flush=True)

all_mounted_files = []
best_ckpts = []
latest_ckpts = []

for s_root in ["/kaggle/input", "/kaggle/working"]:
    if not os.path.exists(s_root):
        continue
    for root, _, files in os.walk(s_root):
        for f in files:
            full = os.path.join(root, f)
            all_mounted_files.append(full)
            f_lower = f.lower()
            if "model7_qwen_best.pt" in f_lower or "model7_best.pt" in f_lower:
                best_ckpts.append(full)
            elif "model7_qwen_latest.pt" in f_lower or "model7_latest.pt" in f_lower:
                latest_ckpts.append(full)

# Fallback checkpoint search if exact naming differs
if not best_ckpts and not latest_ckpts:
    for full in all_mounted_files:
        f_lower = os.path.basename(full).lower()
        if f_lower.endswith(".pt") and ("model7" in f_lower or "best" in f_lower):
            best_ckpts.append(full)

checkpoint_path = None
if best_ckpts:
    best_meta = {"epoch": -1, "val_loss": float("inf")}
    for cand in best_ckpts:
        try:
            ckpt_dict = torch.load(cand, map_location="cpu")
            vl = ckpt_dict.get("val_loss", float("inf"))
            ep = ckpt_dict.get("epoch", 1)
            if vl < best_meta["val_loss"]:
                best_meta = {"epoch": ep, "val_loss": vl}
                checkpoint_path = cand
        except Exception:
            if checkpoint_path is None:
                checkpoint_path = cand
elif latest_ckpts:
    checkpoint_path = latest_ckpts[0]

features_spatial_path = None
manifest_path = None

for full in all_mounted_files:
    fname = os.path.basename(full)
    if fname in ["features_spatial_test.dat", "features_spatial.dat"] and features_spatial_path is None:
        features_spatial_path = full
    elif fname in ["mimic_master_split_with_rag.csv", "mimic_master_split.csv", "mimic_test_impressions.csv"] and manifest_path is None:
        manifest_path = full

print(f"Selected Checkpoint     : {checkpoint_path}", flush=True)
print(f"Found Test Spatial MMAP : {features_spatial_path}", flush=True)
print(f"Found Manifest CSV      : {manifest_path}", flush=True)

if not checkpoint_path or not features_spatial_path or not manifest_path:
    print("\n" + "!" * 75, flush=True)
    print("ERROR: Missing required artifacts! Here are the files currently mounted:", flush=True)
    for p in sorted(all_mounted_files)[:40]:
        print(f"  - {p}", flush=True)
    if len(all_mounted_files) > 40:
        print(f"  ... and {len(all_mounted_files) - 40} more files.", flush=True)
    print("\nPlease verify you attached via '+ Add Input':", flush=True)
    print("  1) Model 7 training run output (containing model7_qwen_best.pt)", flush=True)
    print("  2) Model 7 data prep dataset (containing features_spatial_test.dat & mimic_master_split_with_rag.csv)", flush=True)
    print("!" * 75 + "\n", flush=True)
    raise FileNotFoundError("Missing Model 7 evaluation artifacts in /kaggle/input.")

df_all = pd.read_csv(manifest_path)
if "split" in df_all.columns:
    test_df = df_all[df_all["split"] == "test"].reset_index(drop=True)
else:
    test_df = df_all.reset_index(drop=True)

# Defensive memmap sizing to guarantee 100% alignment without out-of-bounds error
bytes_per_sample = 128 * 768 * 2  # float16 is 2 bytes
mmap_bytes = os.path.getsize(features_spatial_path)
inferred_samples = mmap_bytes // bytes_per_sample
print(f"Test Spatial MMAP size  : {mmap_bytes:,} bytes (~{inferred_samples} samples)", flush=True)

num_test = len(test_df)
if inferred_samples != num_test:
    print(f"Note: Aligning test_df ({num_test}) with MMAP sample count ({inferred_samples}).", flush=True)
    num_test = min(num_test, inferred_samples)
    test_df = test_df.iloc[:num_test].reset_index(drop=True)

if QUICK_TEST:
    quick_n = 20 if isinstance(QUICK_TEST, bool) else int(QUICK_TEST)
    num_test = min(num_test, quick_n)
    test_df = test_df.iloc[:num_test].reset_index(drop=True)
    print(f"\n*** QUICK TEST MODE ACTIVE: Evaluating only first {num_test} samples ***\n", flush=True)

print(f"Ready to evaluate {num_test} test samples for pure Stand-Alone (No-RAG) benchmark.", flush=True)

test_spatial_mmap = np.memmap(
    features_spatial_path,
    dtype="float16",
    mode="r",
    shape=(inferred_samples, 128, 768)
)

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

thresholds_path = None
for full in all_mounted_files:
    if os.path.basename(full) == "optimal_thresholds.json":
        thresholds_path = full
        break

optimal_thresholds = {col: 0.5 for col in CHEXPERT_COLS}
if thresholds_path and os.path.exists(thresholds_path):
    with open(thresholds_path) as f:
        loaded_th = json.load(f)
        optimal_thresholds.update(loaded_th)
    print("Loaded class-specific optimal decision thresholds from JSON:", flush=True)
    for k, v in optimal_thresholds.items():
        print(f"  {k:<26}: {v:.2f}", flush=True)
else:
    print("Using default 0.5 decision thresholds.", flush=True)

# -------------------------------------------------------------------------
# Load Architecture with Perceiver Resampler & Soft Conditioning
# -------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# Check for local mounted Qwen weights in /kaggle/input if available
for root, dirs, _ in os.walk("/kaggle/input"):
    for d in dirs:
        if "qwen2.5-3b-instruct" in d.lower() or "qwen-2.5-3b" in d.lower():
            cand = os.path.join(root, d)
            if any(fname.endswith(".json") for fname in os.listdir(cand)):
                MODEL_NAME = cand
                print(f"Found local Qwen model directory: {MODEL_NAME}", flush=True)
                break
    if MODEL_NAME != "Qwen/Qwen2.5-3B-Instruct":
        break

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if torch.cuda.is_available():
    try:
        print("Attempting to load Qwen with 4-bit BitsAndBytes quantization...", flush=True)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        base_qwen = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            attn_implementation="sdpa",
            device_map={"": 0},
            trust_remote_code=True
        )
        base_qwen = prepare_model_for_kbit_training(base_qwen)
        print("Loaded Qwen in 4-bit NF4 successfully!", flush=True)
    except Exception as bnb_err:
        print(f"Notice: 4-bit quantization could not be initialized ({bnb_err}).", flush=True)
        print("Auto-switching to native torch.float16 on GPU (occupies ~6 GB VRAM)...", flush=True)
        base_qwen = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            attn_implementation="sdpa",
            device_map={"": 0},
            trust_remote_code=True
        )
        print("Loaded Qwen in native float16 directly on GPU successfully!", flush=True)
else:
    base_qwen = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True
    ).to(device)

ckpt = torch.load(checkpoint_path, map_location="cpu")
ckpt_epoch = ckpt.get("epoch", "unknown")
ckpt_loss = ckpt.get("val_loss", "unknown")
print(f"Loaded Checkpoint Metadata: Epoch={ckpt_epoch} | Validation Loss={ckpt_loss}", flush=True)

detected_rank = 32
for k, v in ckpt["lora_state"].items():
    if "lora_A" in k:
        detected_rank = v.shape[0]
        break

detected_alpha = detected_rank * 2
print(f"Auto-Detected LoRA Config: Rank r={detected_rank}, Alpha={detected_alpha}", flush=True)

peft_config = LoraConfig(
    r=detected_rank,
    lora_alpha=detected_alpha,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
# Ensure torchao is bypassed before get_peft_model
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao as _plt
    _plt.is_torchao_available = lambda: False
except Exception:
    pass

qwen_lora_model = get_peft_model(base_qwen, peft_config)

class AttentionPoolClassifier(nn.Module):
    def __init__(self, visual_dim=768, num_classes=14):
        super().__init__()
        self.num_classes = num_classes
        self.visual_dim = visual_dim
        
        self.attn_proj = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.Tanh(),
            nn.Linear(256, num_classes)
        )
        
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(visual_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1)
            ) for _ in range(num_classes)
        ])
        
    def forward(self, spatial_tokens):
        attn_scores = self.attn_proj(spatial_tokens)
        attn_weights = F.softmax(attn_scores, dim=1)
        pooled = torch.einsum('bpd,bpc->bcd', spatial_tokens, attn_weights)
        logits_list = [self.heads[c](pooled[:, c, :]) for c in range(self.num_classes)]
        logits = torch.cat(logits_list, dim=-1)
        return logits, attn_weights

class PerceiverResampler(nn.Module):
    def __init__(self, visual_dim=768, qwen_dim=2048, num_queries=32, num_heads=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_queries = num_queries
        self.qwen_dim = qwen_dim
        
        self.input_proj = nn.Linear(visual_dim, qwen_dim)
        self.input_norm = nn.LayerNorm(qwen_dim)
        self.queries = nn.Parameter(torch.randn(1, num_queries, qwen_dim) * 0.02)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=qwen_dim,
            nhead=num_heads,
            dim_feedforward=qwen_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(qwen_dim)
        
    def forward(self, visual_tokens):
        B = visual_tokens.size(0)
        memory = self.input_norm(self.input_proj(visual_tokens))
        queries = self.queries.expand(B, -1, -1)
        resampled = self.decoder(tgt=queries, memory=memory)
        return self.output_norm(resampled)

class MultiTaskModel7(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_queries=32):
        super().__init__()
        self.qwen = qwen_model
        self.num_queries = num_queries
        self.num_prefix_tokens = num_queries + 1
        self.qwen_dim = qwen_dim
        
        self.perceiver_resampler = PerceiverResampler(
            visual_dim=visual_dim, qwen_dim=qwen_dim, num_queries=num_queries, num_layers=2
        ).to(device=device)
        
        self.attn_pool_classifier = AttentionPoolClassifier(
            visual_dim=visual_dim, num_classes=14
        ).to(device=device)
        
        self.cls_embed_proj = nn.Sequential(
            nn.Linear(14, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, qwen_dim),
            nn.LayerNorm(qwen_dim)
        ).to(device=device)
        
    @torch.inference_mode()
    def generate_report(self, spatial_token, max_new_tokens=150, num_beams=4, **kwargs):
        self.eval()
        self.qwen.config.use_cache = True
        
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
            
        # 1. Step A: Disease Classifier Prediction with Attention Pooling
        cls_logits, attn_weights = self.attn_pool_classifier(spatial_token.float())
        cls_probs_t = torch.sigmoid(cls_logits.float())
        cls_probs = cls_probs_t.squeeze(0).cpu().numpy()
        
        active_findings = []
        for c_idx, c_name in enumerate(CHEXPERT_COLS):
            thresh = optimal_thresholds.get(c_name, 0.5)
            if cls_probs[c_idx] >= thresh:
                active_findings.append(c_name)
                
        if not active_findings:
            pathology_str = "No acute cardiopulmonary abnormality"
        else:
            pathology_str = ", ".join(active_findings)
            
        # 2. Step B: Build Stand-Alone Prompt (Pure No-RAG: Reference: None.)
        prompt_str = f"Pathology Assessment: {pathology_str}\nReference: None.\nImpression:"
        prompt_enc = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_ids = prompt_enc["input_ids"]
        prompt_embeds = self.qwen.get_input_embeddings()(prompt_ids)
        
        # 3. Step C: Multimodal Prefix (32 Perceiver Queries + 1 Soft Disease Token)
        visual_queries = self.perceiver_resampler(spatial_token.float())
        cls_token = self.cls_embed_proj(cls_probs_t).unsqueeze(1)
        
        if visual_queries.dtype != prompt_embeds.dtype:
            visual_queries = visual_queries.to(prompt_embeds.dtype)
        if cls_token.dtype != prompt_embeds.dtype:
            cls_token = cls_token.to(prompt_embeds.dtype)
            
        prefix_embeds = torch.cat([visual_queries, cls_token], dim=1)
        inputs_embeds = torch.cat([prefix_embeds, prompt_embeds], dim=1)
        
        # 4. Step D: Beam Search Decoding (Clinically Safe, Deterministic, Multi-Sentence)
        attn_mask_2d = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)
        
        gen_out = self.qwen.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask_2d,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
            repetition_penalty=1.05,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )
        
        if gen_out.size(1) > inputs_embeds.size(1):
            gen_tokens = gen_out[0, inputs_embeds.size(1):]
        else:
            gen_tokens = gen_out[0]
        pred_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        if pred_text.lower().startswith("impression:"):
            pred_text = pred_text[len("impression:"):].strip()
        
        return pred_text, cls_probs, attn_weights.squeeze(0).cpu().numpy(), pathology_str

def clean_state_dict(sd):
    return {k.replace("module.", ""): v for k, v in sd.items()}

model = MultiTaskModel7(qwen_lora_model).to(device)
model.perceiver_resampler.load_state_dict(clean_state_dict(ckpt["perceiver_resampler"]))
model.cls_embed_proj.load_state_dict(clean_state_dict(ckpt["cls_embed_proj"]))
model.attn_pool_classifier.load_state_dict(clean_state_dict(ckpt["attn_pool_classifier"]))
set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
model.eval()
print(f"Model 7 loaded successfully from checkpoint: {checkpoint_path}", flush=True)

# -------------------------------------------------------------------------
# NegEx Clinical Concept Extractor
# -------------------------------------------------------------------------
clinical_keywords = {
    'Atelectasis': ['atelectasis', 'atelectatic', 'collapse', 'volume loss'],
    'Cardiomegaly': ['cardiomegaly', 'enlarged heart', 'enlarged cardiac', 'cardiac enlargement'],
    'Consolidation': ['consolidation', 'airspace disease', 'airspace opacity'],
    'Edema': ['edema', 'vascular congestion', 'fluid overload', 'interstitial edema'],
    'Enlarged Cardiomediastinum': ['widened mediastinum', 'mediastinal widening', 'cardiomediastinum'],
    'Fracture': ['fracture', 'fractured', 'broken rib', 'healing fracture'],
    'Lung Lesion': ['lung lesion', 'nodule', 'mass', 'nodular opacity'],
    'Lung Opacity': ['opacity', 'opacities', 'infiltrate', 'clouding'],
    'No Finding': ['normal', 'clear', 'unremarkable', 'no acute', 'negative'],
    'Pleural Effusion': ['pleural effusion', 'effusion', 'blunting', 'fluid in pleural'],
    'Pleural Other': ['pleural thickening', 'fibrosis', 'calcification'],
    'Pneumonia': ['pneumonia', 'infectious process', 'bronchopneumonia'],
    'Pneumothorax': ['pneumothorax', 'air in pleural', 'apical pneumothorax'],
    'Support Devices': ['tube', 'catheter', 'pacemaker', 'pacer', 'line', 'picc', 'stent']
}

negation_prefix_pattern = re.compile(r"\b(no|without|free of|negative for|absence of|denies|cleared of)\b\s*([\w\s]{0,35})$", re.IGNORECASE)
negation_suffix_pattern = re.compile(r"^([\w\s]{0,35})\s*\b(ruled out|is absent|resolved|excluded|unlikely|is not seen|not identified|has resolved|is clear)\b", re.IGNORECASE)

def detect_clinical_presence(text, keywords):
    text_lower = text.lower()
    for kw in keywords:
        matches = [m.start() for m in re.finditer(re.escape(kw), text_lower)]
        for match_idx in matches:
            prefix_window = text_lower[max(0, match_idx - 35):match_idx]
            suffix_window = text_lower[match_idx + len(kw):min(len(text_lower), match_idx + len(kw) + 35)]
            if not negation_prefix_pattern.search(prefix_window) and not negation_suffix_pattern.search(suffix_window):
                return 1
    return 0

# -------------------------------------------------------------------------
# Test Benchmark Loop Execution (Pure Stand-Alone No-RAG)
# -------------------------------------------------------------------------
smooth = SmoothingFunction().method1
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

results = []
bleu1_s, bleu2_s, bleu3_s, bleu4_s = [], [], [], []
rouge1_s, rouge2_s, rougeL_s = [], [], []
distinct1_s, distinct2_s = [], []

gt_pathology_list = []
pred_pathology_list = []
corpus_refs = []
corpus_hyps = []

# Collect publication-grade XAI samples
xai_samples = []

start_eval_time = time.time()
print(f"Starting Model 7 Pure Stand-Alone (No-RAG) Benchmark on {num_test} cases (Beam Search k=4)...", flush=True)

for i in range(num_test):
    row = test_df.iloc[i]
    orig_idx = int(row["orig_split_idx"]) if ("orig_split_idx" in row and pd.notna(row["orig_split_idx"])) else i
    if orig_idx >= inferred_samples:
        orig_idx = i % inferred_samples

    spatial_t = torch.from_numpy(np.array(test_spatial_mmap[orig_idx])).to(device)
    
    # Safely get ground truth report from possible column names
    gt_cand = row.get("report", row.get("impression", ""))
    gt_text = str(gt_cand).strip() if pd.notna(gt_cand) and str(gt_cand).strip() else "Normal chest radiograph."
    
    # Pure No-RAG generation: Reference is strictly "None."
    pred_text, cls_probs, attn_maps, assessed_str = model.generate_report(
        spatial_t, max_new_tokens=150, num_beams=4
    )
    
    study_id = int(row["study_id"]) if ("study_id" in row and pd.notna(row["study_id"])) else i
    dicom_id = str(row["dicom_id"]) if ("dicom_id" in row and pd.notna(row["dicom_id"])) else f"case_{i}"

    # Save candidates for publication case study
    if len(xai_samples) < 8:
        xai_samples.append({
            "idx": i,
            "study_id": study_id,
            "dicom_id": dicom_id,
            "gt_text": gt_text,
            "pred_text": pred_text,
            "assessed_pathologies": assessed_str,
            "cls_probs": cls_probs,
            "attn_maps": attn_maps
        })
    
    ref_tok = [nltk.word_tokenize(gt_text.lower())] if gt_text else [[""]]
    hyp_tok = nltk.word_tokenize(pred_text.lower()) if pred_text else [""]
    
    corpus_refs.append(ref_tok)
    corpus_hyps.append(hyp_tok)
    
    b1 = sentence_bleu(ref_tok, hyp_tok, weights=(1, 0, 0, 0), smoothing_function=smooth)
    b2 = sentence_bleu(ref_tok, hyp_tok, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    b3 = sentence_bleu(ref_tok, hyp_tok, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    b4 = sentence_bleu(ref_tok, hyp_tok, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    
    rouge_res = rouge_evaluator.score(gt_text, pred_text)
    r1 = rouge_res['rouge1'].fmeasure
    r2 = rouge_res['rouge2'].fmeasure
    rL = rouge_res['rougeL'].fmeasure
    
    bleu1_s.append(b1)
    bleu2_s.append(b2)
    bleu3_s.append(b3)
    bleu4_s.append(b4)
    rouge1_s.append(r1)
    rouge2_s.append(r2)
    rougeL_s.append(rL)
    
    if len(hyp_tok) >= 1:
        distinct1_s.append(len(set(hyp_tok)) / len(hyp_tok))
    else:
        distinct1_s.append(1.0)
        
    if len(hyp_tok) >= 2:
        bigrams = list(ngrams(hyp_tok, 2))
        distinct2_s.append(len(set(bigrams)) / len(bigrams))
    else:
        distinct2_s.append(1.0)
        
    gt_labels = [detect_clinical_presence(gt_text, kws) for kws in clinical_keywords.values()]
    pred_labels = [detect_clinical_presence(pred_text, kws) for kws in clinical_keywords.values()]
    
    gt_pathology_list.append(gt_labels)
    pred_pathology_list.append(pred_labels)
    
    results.append({
        "study_id": study_id,
        "dicom_id": dicom_id,
        "ground_truth": gt_text,
        "prediction": pred_text,
        "bleu1": round(b1, 4),
        "bleu4": round(b4, 4),
        "rougeL": round(rL, 4)
    })
    
    if (i + 1) % 100 == 0 or (i + 1) == num_test:
        elapsed = (time.time() - start_eval_time) / 60.0
        pct = ((i + 1) / num_test) * 100.0
        eta = (elapsed / (i + 1)) * (num_test - (i + 1))
        avg_b1 = np.mean(bleu1_s)
        avg_rl = np.mean(rougeL_s)
        print(f"[Model 7 No-RAG Eval {i+1:04d}/{num_test} ({pct:5.1f}%)] Elapsed: {elapsed:.1f}m | ETA: {eta:.1f}m | BLEU-1: {avg_b1:.4f} | ROUGE-L: {avg_rl:.4f}", flush=True)
        
    if (i + 1) % 500 == 0:
        torch.cuda.empty_cache()
        
    # Periodic checkpoint save every 1000 samples to safeguard against disconnection
    if (i + 1) % 1000 == 0:
        pd.DataFrame(results).to_csv("/kaggle/working/model7_no_rag_checkpoint_predictions.csv", index=False)

# -------------------------------------------------------------------------
# Free Generator Model from GPU to Maximize VRAM for Semantic Evaluation
# -------------------------------------------------------------------------
print("\nFreeing generator model from GPU VRAM to ensure zero-risk BERTScore & SBERT execution...", flush=True)
del model, base_qwen, qwen_lora_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# Clinical Diagnostics & Aggregate NLP Metrics
# -------------------------------------------------------------------------
y_true = np.array(gt_pathology_list)
y_pred = np.array(pred_pathology_list)

pathology_names = list(clinical_keywords.keys())

core10_indices = [i for i, name in enumerate(pathology_names) if name not in [
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Pleural Other'
]]

macro_f1_all14 = f1_score(y_true, y_pred, average="macro", zero_division=0)
macro_f1_core10 = f1_score(y_true[:, core10_indices], y_pred[:, core10_indices], average="macro", zero_division=0)

micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
micro_precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
micro_recall = recall_score(y_true, y_pred, average="micro", zero_division=0)

corpus_b1 = corpus_bleu(corpus_refs, corpus_hyps, weights=(1, 0, 0, 0))
corpus_b4 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.25, 0.25, 0.25, 0.25))

clinical_diagnostic_table = []
for idx, name in enumerate(pathology_names):
    yt = y_true[:, idx]
    yp = y_pred[:, idx]
    
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1 = f1_score(yt, yp, average="binary", zero_division=0)
    
    clinical_diagnostic_table.append({
        "Pathology": name,
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Sensitivity": round(sensitivity, 4),
        "Specificity": round(specificity, 4),
        "PPV": round(ppv, 4),
        "NPV": round(npv, 4),
        "F1": round(f1, 4)
    })

# -------------------------------------------------------------------------
# Semantic Evaluation: Clinical BERTScore & SBERT Cosine Similarity
# -------------------------------------------------------------------------
print("\n" + "=" * 70, flush=True)
print("Computing Modern Clinical Semantic Metrics (BERTScore & SBERT)", flush=True)
print("=" * 70, flush=True)

all_preds = [str(r["prediction"]).strip() if str(r["prediction"]).strip() else "No acute abnormality." for r in results]
all_refs = [str(r["ground_truth"]).strip() if str(r["ground_truth"]).strip() else "Normal chest radiograph." for r in results]

clinical_bert_f1 = 0.0
clinical_bert_precision = 0.0
clinical_bert_recall = 0.0

try:
    print("1. Computing Clinical BERTScore with Microsoft PubMedBERT...", flush=True)
    from bert_score import score as bert_score_eval
    P_b, R_b, F1_b = bert_score_eval(
        all_preds, all_refs,
        model_type="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
        num_layers=12,
        device=str(device),
        batch_size=64,
        verbose=False
    )
    clinical_bert_f1 = float(F1_b.mean().item())
    clinical_bert_precision = float(P_b.mean().item())
    clinical_bert_recall = float(R_b.mean().item())
    print(f"  -> Clinical BERTScore F1 (PubMedBERT) : {clinical_bert_f1:.4f}", flush=True)
    print(f"  -> Clinical BERTScore Rec (PubMedBERT): {clinical_bert_recall:.4f}", flush=True)
except Exception as e:
    print(f"  Warning: PubMedBERT encountered ({e}). Trying BioBERT fallback...", flush=True)
    try:
        from bert_score import score as bert_score_eval
        P_b, R_b, F1_b = bert_score_eval(
            all_preds, all_refs,
            model_type="dmis-lab/biobert-v1.1",
            num_layers=12,
            device=str(device),
            batch_size=64,
            verbose=False
        )
        clinical_bert_f1 = float(F1_b.mean().item())
        clinical_bert_precision = float(P_b.mean().item())
        clinical_bert_recall = float(R_b.mean().item())
        print(f"  -> BioBERT F1: {clinical_bert_f1:.4f}", flush=True)
    except Exception as e2:
        print(f"  Warning: BioBERT encountered ({e2}). Trying DistilBERT fallback...", flush=True)
        try:
            from bert_score import score as bert_score_eval
            P_b, R_b, F1_b = bert_score_eval(
                all_preds, all_refs,
                model_type="distilbert-base-uncased",
                num_layers=6,
                device=str(device),
                batch_size=64,
                verbose=False
            )
            clinical_bert_f1 = float(F1_b.mean().item())
            clinical_bert_precision = float(P_b.mean().item())
            clinical_bert_recall = float(R_b.mean().item())
            print(f"  -> DistilBERT F1: {clinical_bert_f1:.4f}", flush=True)
        except Exception as e3:
            print(f"  Warning: All BERTScore fallbacks failed ({e3}).", flush=True)

semantic_cosine_sim = 0.0
try:
    print("2. Computing Sentence Embedding Semantic Cosine Similarity (SBERT)...", flush=True)
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))
    emb_preds = sbert.encode(all_preds, convert_to_tensor=True, batch_size=128)
    emb_refs = sbert.encode(all_refs, convert_to_tensor=True, batch_size=128)
    cos_sims = F.cosine_similarity(emb_preds, emb_refs, dim=1).cpu().numpy()
    semantic_cosine_sim = float(np.mean(cos_sims))
    print(f"  -> Semantic Cosine Similarity (SBERT) : {semantic_cosine_sim:.4f}", flush=True)
    del sbert, emb_preds, emb_refs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as e:
    print(f"  Warning: SBERT computation encountered: {e}. Using fallback.", flush=True)

# -------------------------------------------------------------------------
# Compile Summary Metrics
# -------------------------------------------------------------------------
summary_metrics = {
    "Evaluation_Mode": "Stand-Alone No-RAG (Reference: None.)",
    "Dataset": "MIMIC-CXR Internal Test Set",
    "N_Samples": int(num_test),
    "Sentence BLEU-1": float(np.mean(bleu1_s)),
    "Sentence BLEU-2": float(np.mean(bleu2_s)),
    "Sentence BLEU-3": float(np.mean(bleu3_s)),
    "Sentence BLEU-4": float(np.mean(bleu4_s)),
    "Corpus BLEU-1 (NIST)": float(corpus_b1),
    "Corpus BLEU-4 (NIST)": float(corpus_b4),
    "ROUGE-1": float(np.mean(rouge1_s)),
    "ROUGE-2": float(np.mean(rouge2_s)),
    "ROUGE-L": float(np.mean(rougeL_s)),
    "Distinct-1": float(np.mean(distinct1_s)),
    "Distinct-2": float(np.mean(distinct2_s)),
    "Clinical_Macro_F1_All14": float(macro_f1_all14),
    "Clinical_Macro_F1_Core10": float(macro_f1_core10),
    "Clinical_Micro_F1": float(micro_f1),
    "Clinical_Micro_Precision": float(micro_precision),
    "Clinical_Micro_Recall": float(micro_recall),
    "Clinical_BERTScore_F1": float(clinical_bert_f1),
    "Clinical_BERTScore_Precision": float(clinical_bert_precision),
    "Clinical_BERTScore_Recall": float(clinical_bert_recall),
    "Semantic_Cosine_Similarity_SBERT": float(semantic_cosine_sim),
    "Evaluated_Checkpoint": str(checkpoint_path),
    "Evaluation_Duration_Minutes": float((time.time() - start_eval_time) / 60.0)
}

print("\n" + "="*70, flush=True)
print("FINAL MODEL 7 STAND-ALONE (NO-RAG) BENCHMARK RESULTS", flush=True)
print("="*70, flush=True)
for k, v in summary_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<34}: {v:.4f}", flush=True)
    else:
        print(f"  {k:<34}: {v}", flush=True)

# -------------------------------------------------------------------------
# Publication Heatmaps: 14-Panel Confusion Matrix
# -------------------------------------------------------------------------
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for idx, name in enumerate(pathology_names):
    cm = np.array([
        [clinical_diagnostic_table[idx]["TN"], clinical_diagnostic_table[idx]["FP"]],
        [clinical_diagnostic_table[idx]["FN"], clinical_diagnostic_table[idx]["TP"]]
    ])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[idx],
                xticklabels=["Pred Neg", "Pred Pos"], yticklabels=["Actual Neg", "Actual Pos"])
    axes[idx].set_title(f"{name}\n(Sens: {clinical_diagnostic_table[idx]['Sensitivity']:.2f} | F1: {clinical_diagnostic_table[idx]['F1']:.2f})", fontsize=10, fontweight="bold")

for j in range(len(pathology_names), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Model 7 (Stand-Alone No-RAG): 14 Clinical Pathology Confusion Matrices", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig("/kaggle/working/model7_no_rag_pathology_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.close()
print("14-Panel Confusion Matrix Heatmap saved to /kaggle/working/model7_no_rag_pathology_confusion_matrices.png", flush=True)

# -------------------------------------------------------------------------
# Explainable AI (XAI): 16x8 True Anatomical Spatial Heatmaps
# -------------------------------------------------------------------------
print("Generating Explainable AI Attention Pooling Spatial Heatmaps (16x8 True 2D Grid)...", flush=True)
if xai_samples:
    fig, axes = plt.subplots(len(xai_samples), 4, figsize=(14, 4.5 * len(xai_samples)))
    if len(xai_samples) == 1:
        axes = np.expand_dims(axes, 0)
        
    focus_diseases = ["Cardiomegaly", "Pleural Effusion", "Pneumonia", "Atelectasis"]
    focus_indices = [pathology_names.index(d) for d in focus_diseases]
    
    for row_idx, sample in enumerate(xai_samples):
        for col_idx, d_idx in enumerate(focus_indices):
            grid_attn = sample["attn_maps"][:, d_idx].reshape(16, 8)
            grid_norm = (grid_attn - grid_attn.min()) / (grid_attn.max() - grid_attn.min() + 1e-8)
            prob_score = sample["cls_probs"][d_idx]
            
            sns.heatmap(grid_norm, cmap="magma", cbar=False, ax=axes[row_idx, col_idx])
            axes[row_idx, col_idx].set_title(f"{focus_diseases[col_idx]} (P={prob_score:.2f})", fontsize=10, fontweight="bold")
            axes[row_idx, col_idx].axis("off")
            
    plt.suptitle("Model 7 (Stand-Alone): Explainable AI True 2D Spatial Heatmaps (16x8 Grid)", fontsize=14, fontweight="bold", y=0.99)
    plt.tight_layout()
    plt.savefig("/kaggle/working/model7_no_rag_pathology_attention_heatmaps.png", dpi=300, bbox_inches="tight")
    plt.close()
    print("XAI Heatmaps saved to /kaggle/working/model7_no_rag_pathology_attention_heatmaps.png", flush=True)

# -------------------------------------------------------------------------
# Qualitative Case Studies Export
# -------------------------------------------------------------------------
sorted_cases = sorted(results, key=lambda x: x["bleu4"], reverse=True)
top_perfect_cases = sorted_cases[:3]

discrepancies = []
for i in range(num_test):
    gt_sum = sum(y_true[i])
    pred_sum = sum(y_pred[i])
    diff = abs(gt_sum - pred_sum)
    if diff >= 2:
        discrepancies.append({
            "study_id": int(results[i]["study_id"]),
            "ground_truth": results[i]["ground_truth"],
            "prediction": results[i]["prediction"],
            "gt_active": [pathology_names[j] for j in range(14) if y_true[i, j] == 1],
            "pred_active": [pathology_names[j] for j in range(14) if y_pred[i, j] == 1],
            "bleu4": results[i]["bleu4"]
        })
    if len(discrepancies) >= 3:
        break

def to_serializable(val):
    if hasattr(val, "item"):
        return val.item()
    if isinstance(val, (np.integer, np.int64)):
        return int(val)
    if isinstance(val, (np.floating, np.float32, np.float64)):
        return float(val)
    if isinstance(val, np.ndarray):
        return val.tolist()
    return str(val)

qualitative_export = {
    "Top_Perfect_Matches": top_perfect_cases,
    "Clinical_Discrepancies": discrepancies
}
with open("/kaggle/working/model7_no_rag_qualitative_case_studies.json", "w") as f:
    json.dump(qualitative_export, f, indent=2, default=to_serializable)

print("Qualitative Case Studies saved to /kaggle/working/model7_no_rag_qualitative_case_studies.json", flush=True)

# -------------------------------------------------------------------------
# Save All Output Artifacts
# -------------------------------------------------------------------------
pred_df = pd.DataFrame(results)
diag_df = pd.DataFrame(clinical_diagnostic_table)

pred_df.to_csv("/kaggle/working/model7_no_rag_test_predictions.csv", index=False)
pred_df.to_csv("/kaggle/working/model7_test_predictions_no_rag.csv", index=False)

diag_df.to_csv("/kaggle/working/model7_no_rag_clinical_diagnostic_metrics.csv", index=False)

with open("/kaggle/working/model7_no_rag_test_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=2, default=to_serializable)
with open("/kaggle/working/model7_test_metrics_no_rag.json", "w") as f:
    json.dump(summary_metrics, f, indent=2, default=to_serializable)

# LaTeX Table Generation
b1_str = f"{summary_metrics['Sentence BLEU-1']:.4f}"
b4_str = f"{summary_metrics['Sentence BLEU-4']:.4f}"
r1_str = f"{summary_metrics['ROUGE-1']:.4f}"
rl_str = f"{summary_metrics['ROUGE-L']:.4f}"
d2_str = f"{summary_metrics['Distinct-2']:.4f}"
f1_all = f"{summary_metrics['Clinical_Macro_F1_All14']:.4f}"
f1_cor = f"{summary_metrics['Clinical_Macro_F1_Core10']:.4f}"
f1_mic = f"{summary_metrics['Clinical_Micro_F1']:.4f}"
rc_mic = f"{summary_metrics['Clinical_Micro_Recall']:.4f}"

latex_lines = [
    r"\begin{table*}[htbp]",
    r"\centering",
    r"\caption{Model 7 Stand-Alone (No-RAG) Evaluation on the MIMIC-CXR Test Cohort}",
    r"\label{tab:model7_no_rag_eval}",
    r"\resizebox{\textwidth}{!}{",
    r"\begin{tabular}{lccccccccc}",
    r"\hline",
    r"\textbf{Model Configuration} & \textbf{BLEU-1} & \textbf{BLEU-4} & \textbf{ROUGE-1} & \textbf{ROUGE-L} & \textbf{Distinct-2} & \textbf{Macro F1 (All14)} & \textbf{Macro F1 (Core10)} & \textbf{Micro F1} & \textbf{Micro Rec} \\",
    r"\hline",
    rf"\textbf{{Model 7 (Stand-Alone No-RAG, Ours)}} & \textbf{{{b1_str}}} & \textbf{{{b4_str}}} & \textbf{{{r1_str}}} & \textbf{{{rl_str}}} & \textbf{{{d2_str}}} & \textbf{{{f1_all}}} & \textbf{{{f1_cor}}} & \textbf{{{f1_mic}}} & \textbf{{{rc_mic}}} \\",
    r"\hline",
    r"\end{tabular}",
    r"}",
    r"\end{table*}"
]
with open("/kaggle/working/model7_no_rag_thesis_comparison_table.tex", "w") as f:
    f.write("\n".join(latex_lines))

print("\n" + "="*80, flush=True)
print("MODEL 7 STAND-ALONE (NO-RAG) BENCHMARK COMPLETED SUCCESSFULLY!")
print("Saved files in /kaggle/working:")
print("  - model7_no_rag_test_metrics.json (and model7_test_metrics_no_rag.json)")
print("  - model7_no_rag_test_predictions.csv (and model7_test_predictions_no_rag.csv)")
print("  - model7_no_rag_clinical_diagnostic_metrics.csv")
print("  - model7_no_rag_pathology_confusion_matrices.png")
print("  - model7_no_rag_pathology_attention_heatmaps.png")
print("  - model7_no_rag_qualitative_case_studies.json")
print("  - model7_no_rag_thesis_comparison_table.tex")
print("="*80, flush=True)


Evaluation Compute Device: cuda
Scanning mounted input directories in /kaggle/input...
Selected Checkpoint     : /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/model7_qwen_best.pt
Found Test Spatial MMAP : /kaggle/input/datasets/kimiasafaralian/mimic-cxr-model-7-prep-data/features_spatial_test.dat
Found Manifest CSV      : /kaggle/input/datasets/kimiasafaralian/mimic-cxr-model-7-prep-data/mimic_master_split_with_rag.csv
Loaded 6388 test samples for comprehensive benchmark evaluation.
Loaded class-specific optimal decision thresholds from JSON:
  Atelectasis               : 0.60
  Cardiomegaly              : 0.58
  Consolidation             : 0.58
  Edema                     : 0.62
  Enlarged Cardiomediastinum: 0.60
  Fracture                  : 0.58
  Lung Lesion               : 0.64
  Lung Opacity              : 0.56
  No Finding                : 0.56
  Pleural Effusion          : 0.62
  Pleural Other             : 0.62
  Pneumonia                 : 0.56
  Pneumothor

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Checkpoint Metadata: Epoch=2 | Validation Loss=1.3587450467541005
Auto-Detected LoRA Config: Rank r=32, Alpha=64
Model 7 loaded successfully from checkpoint: /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/model7_qwen_best.pt
Starting Model 7 Benchmark Evaluation on all 6388 test cases (Beam Search k=4)...


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1306: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1314: UserWarning: Passing `no_repeat_ngram_size` with `inputs_embeds` and without `input_ids` to `generate` will apply n-gram constraints only to newly generated tokens, not to the prompt.
  warnings.warn(


[Model 7 Eval 0100/6388 (  1.6%)] Elapsed: 6.5m | ETA: 408.1m | BLEU-1: 0.2080 | ROUGE-L: 0.2016
[Model 7 Eval 0200/6388 (  3.1%)] Elapsed: 10.2m | ETA: 316.1m | BLEU-1: 0.2166 | ROUGE-L: 0.2193
[Model 7 Eval 0300/6388 (  4.7%)] Elapsed: 16.5m | ETA: 334.5m | BLEU-1: 0.2589 | ROUGE-L: 0.2669
[Model 7 Eval 0400/6388 (  6.3%)] Elapsed: 22.8m | ETA: 341.6m | BLEU-1: 0.2804 | ROUGE-L: 0.2853
[Model 7 Eval 0500/6388 (  7.8%)] Elapsed: 28.3m | ETA: 333.5m | BLEU-1: 0.2680 | ROUGE-L: 0.2801
[Model 7 Eval 0600/6388 (  9.4%)] Elapsed: 33.2m | ETA: 320.2m | BLEU-1: 0.2787 | ROUGE-L: 0.2880
[Model 7 Eval 0700/6388 ( 11.0%)] Elapsed: 40.3m | ETA: 327.6m | BLEU-1: 0.2813 | ROUGE-L: 0.2884
[Model 7 Eval 0800/6388 ( 12.5%)] Elapsed: 45.9m | ETA: 320.4m | BLEU-1: 0.2973 | ROUGE-L: 0.3026
[Model 7 Eval 0900/6388 ( 14.1%)] Elapsed: 52.2m | ETA: 318.4m | BLEU-1: 0.2958 | ROUGE-L: 0.3007
[Model 7 Eval 1000/6388 ( 15.7%)] Elapsed: 57.7m | ETA: 310.7m | BLEU-1: 0.2884 | ROUGE-L: 0.2939
[Model 7 Eval 1100/63